# 03_langgraph_agent_test.ipynb
Testing LangGraph agents for narrative generation and validation with diverse scenarios.

In [ ]:
import os
import sys

# Add project root to sys.path
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_PATH not in sys.path:
    sys.path.append(ROOT_PATH)

print(f"Project Root: {ROOT_PATH}")

from src.narrative_agent import build_narrative_graph
from src.constraint_solver import NarrativeConstraintSolver

### 1. LangGraph workflow Visualization
Check the internal structure of the compiled graph.

In [ ]:
graph = build_narrative_graph()

try:
    print("Narrative Agent Workflow (ASCII):")
    graph.get_graph().print_ascii()
except Exception as e:
    print("ASCII Visualization not available (grandalf might be missing).")
    # Attempt to use Mermaid if in a supported environment
    try:
        from IPython.display import Image, display
        # requires pygraphviz or similar for mermaid rendering to png sometimes, 
        # but draw_mermaid_png is a langgraph built-in that might work depending on env
        display(Image(graph.get_graph().draw_mermaid_png()))
    except:
        print("Mermaid visualization also not available.")

### 2. Multi-step Generation Loop (Propp)
Simulate a 4-step story generation using the LangGraph agent.

In [ ]:
def run_narrative_session(theory="propp", steps=4):
    print(f"--- Starting {theory.upper()} Session ({steps} steps) ---")
    state = {
        "current_node_id": None,
        "history": [],
        "world_constants": {},
        "characters": [],
        "theory_type": theory
    }
    
    for i in range(steps):
        state = graph.invoke(state)
        print(f"Step {i+1}: Node={state['current_node_id']}, History={state['history']}")
        if not state['current_node_id']:
            print("Stopped: No valid next node found.")
            break
    
    return state

propp_final_state = run_narrative_session("propp", 4)

### 3. Theory Comparison (Vogler vs Propp)
Verify that the agent respects the chosen narrative theory.

In [ ]:
vogler_final_state = run_narrative_session("vogler", 3)

print("\nFinal Comparison:")
print(f"Propp History: {propp_final_state['history']}")
print(f"Vogler History: {vogler_final_state['history']}")

# Basic validation
assert all(h.startswith('P') for h in propp_final_state['history'])
assert all(h.startswith('V') for h in vogler_final_state['history'])
print("\nTheories correctly respected.")

### 4. Direct CP-SAT Solver Test
Test the `solve_sequence` method which avoids incremental errors by looking ahead.

In [ ]:
solver = NarrativeConstraintSolver(theory_type="propp")
sequence = solver.solve_sequence(length=5)
print(f"Propp Sequence (Look-ahead): {sequence}")

v_solver = NarrativeConstraintSolver(theory_type="vogler")
v_sequence = v_solver.solve_sequence(length=3)
print(f"Vogler Sequence (Look-ahead): {v_sequence}")

### 5. Edge Case: Continuation from mid-story
Provide a pre-existing history and current node to see if the agent picks up correctly.

In [ ]:
mid_state = {
    "current_node_id": "P03",
    "history": ["P01", "P02", "P03"],
    "world_constants": {},
    "characters": [],
    "theory_type": "propp"
}

print("Continuing from P03...")
next_state = graph.invoke(mid_state)
print(f"Next Node: {next_state['current_node_id']}")
print(f"Updated History: {next_state['history']}")

assert next_state['current_node_id'] == "P04"
assert next_state['history'] == ["P01", "P02", "P03", "P04"]